# Notes

In [10]:
import sys
import os
import pandas as pd

import numpy as np
from scipy import stats

import librosa
import librosa.display

from data_engineering import process_features
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [11]:
# Obtener la ruta del directorio raíz
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) 

In [ ]:
familias = os.listdir('songs/')
for familia in familias:
    generos_path = f'songs/{familia}/'
    generos = os.listdir(generos_path)
    
    for genero in generos:
        especies_path = f'{generos_path}{genero}/'
        if os.path.isdir(especies_path):
            especies = [e for e in os.listdir(especies_path) if os.path.isdir(os.path.join(especies_path, e))]
            
            for especie in especies:
                name_comun_path = f'{especies_path}{especie}/'
                name_comuns = [nc for nc in os.listdir(name_comun_path) if os.path.isdir(os.path.join(name_comun_path, nc))]
                
                for name_comun in name_comuns:
                    ruta_base = f'{name_comun_path}{name_comun}'
                    nombre_archivo = f"./data/{familia}_{genero}_{especie}.csv"
                    if not os.path.exists(nombre_archivo):
                        print(f'Procesando: {nombre_archivo}')
                        print(f'\t\t\tRuta base: {ruta_base}')
                        process_features(
                        f"{ruta_base}/",
                        "./data/"
                        f"{familia}_{genero}_{especie}"
                        )
                    else:
                        #pass
                        print(f'El archivo {nombre_archivo} ya existe, saltando procesamiento.')

In [2]:
# process_features(
#     "songs/Alaudidae/Eremophila/Eremophila alpestris/HornedLark/",
#     "./data/"
#     "ElegantWoodcreeper"
# )

In [6]:
ruta_features = './data/'
archivos_csv = os.listdir(ruta_features)
dfs = []
for archivo_csv in archivos_csv:
    # Extraer familia, género y especie del nombre del archivo
    partes_nombre = archivo_csv.split('_')
    familia = partes_nombre[0]
    genero = partes_nombre[1]
    especie = '_'.join(partes_nombre[2:]).replace('.csv', '')  # Asume que el nombre de la especie puede tener guiones bajos

    df = pd.read_csv(os.path.join(ruta_features, archivo_csv))
    # Agregar las columnas con la información extraída
    df['Family'] = familia
    df['Genus'] = genero
    df['Especie'] = especie

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)
df_final.to_csv('data/species_features.csv', index=False)

In [22]:
import polars as pl

In [ ]:
%time
# carga un archivo csv usando polars, visualiza todas las columnas
df = pl.read_csv('data/species_features.csv', has_header=True)
df.select(pl.col("*")).head()

In [70]:
tiranni = [
    'Philepittidae', 'Eurylaimidae', 'Calyptomenidae', 'Sapayoidae', 'Pittidae',
    'Pipridae', 'Cotingidae', 'Tityridae', 'Tyrannidae', 'Melanopareiidae', 'Conopophagidae', 
    'Thamnophilidae', 'Grallariidae', 'Rhinocryptidae', 'Formicariidae', 'Furnariidae',
]

passeri = [ 
        'Menuridae', 'Atrichornithidae', 'Climacteridae', 'Maluridae', 'Meliphagidae', 'Pardalotidae',
        'Petroicidae', 'Orthonychidae', 'Pomatostomidae', 'Cinclosomatidae', 'Neosittidae',
        'Pachycephalidae', 'Dicruridae', 'Oriolidae', 'Artamidae', 'Paradisaeidae', 'Corvidae', 
        'Cyanocorax', 'Corcoracidae', 'Irenidae', 'Laniidae', 'Vireonidae', 'Ptilonorhynchidae', 
        'Turnagridae', 'Callaeidae', 'Alaudidae', 'Chloropseidae', 'Aegithinidae', 'Picathartidae',
        'Bombycillidae', 'Ptilogonatidae', 'Cinclidae', 'Motacillidae', 'Prunellidae', 'Melanocharitidae',
        'Paramythiidae', 'Passeridae', 'Estrildidae', 'Ploceidae', 'Parulidae', 'Thraupidae', 
        'Peucedramidae', 'Fringillidae', 'Cardinalidae', 'Drepanididae', 'Emberizidae', 'Nectariniidae',
        'Dicaeidae', 'Mimidae', 'Sittidae', 'Certhiidae', 'Troglodytidae', 'Polioptilidae', 'Paridae',
        'Aegithalidae', 'Hirundinidae', 'Regulidae', 'Pycnonotidae', 'Sylviidae', 'Acrocephalidae', 
        'Locustellidae', 'Cettiidae', 'Phylloscopidae', 'Cisticolidae', 'Hypocoliidae', 'Zosteropidae',
        'Paradoxornithidae', 'Timaliidae', 'Leiothrichidae', 'Muscicapidae', 'Turdidae', 'Sturnidae',
        'Icteridae'
]

In [71]:
# def suborder(row):
#     if row['Familia'] in tiranni:
#         return 'Tyranni'
#     elif row['Familia'] in passeri:
#         return 'Passeri'
#     else:
#         return 'No definido'

In [73]:
# Aplica la lógica usando expresiones de polars
df = df.with_columns(
    pl.when(pl.col('Familia').is_in(tiranni)).then(pl.lit('Tyranni'))
    .otherwise(
        pl.when(pl.col('Familia').is_in(passeri)).then(pl.lit('Passeri'))
        .otherwise(pl.lit(None))
    ).alias('Suborder')
)

In [ ]:
df.select(pl.col("*")).head()

In [ ]:
# Contar el número de registros por suborder
df.group_by('Suborder').agg([
    pl.col('Suborder').count().alias('count')
])

In [77]:
df.write_csv('data/species_features.csv')